In [1]:
# # This file is run on Google Colab GPU.
# Colab setup — run this first
# torch / transformers are pre-installed on Colab, nothing extra needed

# Combination Model: Multi-Headline FinBERT Aggregation → Tabular Transformer

**Key upgrade over combination_transformer.ipynb:**  
Instead of running FinBERT on the joined headline string, each headline is scored individually.  
Per-day aggregation produces 7 sentiment features: mean, max, variance, count.  
**Total: 11 features** (7 sentiment + 4 price).  
**Target:** intraday direction (close ≥ open).

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import os, gc, shutil
from google.colab import files

WORK_DIR        = "/content"
DATA_FILE       = f"{WORK_DIR}/sp500_headlines_with_prices sep ~.csv"
CACHE_FILE      = f"{WORK_DIR}/finbert_probs_multisent.npy"  # separate cache

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda


In [3]:
print(f"Please upload the data file: {DATA_FILE}")
uploaded = files.upload()

# Rename the uploaded file if necessary (it might be uploaded with a different name if it contains spaces or special characters)
# This assumes the user uploads only one file and it's the correct one.
# If the uploaded file name is different from DATA_FILE, you might need to adjust this.
if uploaded:
    for fn in uploaded.keys():
        if fn != os.path.basename(DATA_FILE):
            print(f"Renaming '{fn}' to '{os.path.basename(DATA_FILE)}'")
            os.rename(fn, os.path.basename(DATA_FILE))
        print(f"File '{os.path.basename(DATA_FILE)}' uploaded successfully.")

Please upload the data file: /content/sp500_headlines_with_prices sep ~.csv


Saving sp500_headlines_with_prices sep ~.csv to sp500_headlines_with_prices sep ~.csv
File 'sp500_headlines_with_prices sep ~.csv' uploaded successfully.


In [4]:
# 1. Load data & target
df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["target"] = (df["close_price"] >= df["open_price"]).astype(int)

# Filter out near-flat days: only keep days with meaningful intraday movement
# These days have ambiguous direction and add noise to the classifier
MOVE_THRESHOLD = 0.002  # 0.2% minimum |close - open| / open
df["daily_move"] = (df["close_price"] - df["open_price"]).abs() / df["open_price"]
before = len(df)
df = df[df["daily_move"] > MOVE_THRESHOLD].reset_index(drop=True)

print(f"Total rows: {before} → {len(df)} after removing near-flat days (<{MOVE_THRESHOLD:.1%} move)")
print(f"Removed: {before - len(df)} rows ({(before-len(df))/before:.1%})")
print(f"Target distribution: {df['target'].value_counts().to_dict()}")

Total rows: 3507 → 2572 after removing near-flat days (<0.2% move)
Removed: 935 rows (26.7%)
Target distribution: {1: 1425, 0: 1147}


In [5]:
# 2. FinBERT sentiment (cached)
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert   = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
finbert.eval()

def get_finbert_probs(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt",
                           truncation=True, max_length=512, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = finbert(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

if os.path.exists(CACHE_FILE):
    print("Loading cached FinBERT probs...")
    finbert_probs = np.load(CACHE_FILE)
else:
    print("Running FinBERT inference...")
    texts = df["combined_headline"].fillna("neutral").tolist()
    finbert_probs = get_finbert_probs(texts)
    np.save(CACHE_FILE, finbert_probs)
    print(f"Saved to {CACHE_FILE}")

# {0: positive, 1: negative, 2: neutral}
df["sent_pos"] = finbert_probs[:, 0]
df["sent_neg"] = finbert_probs[:, 1]
df["sent_neu"] = finbert_probs[:, 2]

# Free GPU/MPS memory before the rest
del finbert, tokenizer
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("FinBERT features added. Model freed from memory.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running FinBERT inference...
Saved to /content/finbert_probs_multisent.npy
FinBERT features added. Model freed from memory.


In [6]:
# 3. Feature engineering — price features (no lookahead leakage)
df["gap_open"]    = (df["open_price"] - df["close_price"].shift(1)) / df["close_price"].shift(1)
df["_ma5"]        = df["close_price"].rolling(5).mean()
df["_vol_ma5"]    = df["volume"].rolling(5).mean()

def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs  = avg_gain / (avg_loss + 1e-9)
    return (100 - 100 / (1 + rs)) / 100.0

df["_rsi14"] = compute_rsi(df["close_price"], 14)
df["prev_ma5_ratio"] = df["close_price"].shift(1) / df["_ma5"].shift(1)
df["prev_rsi14"]     = df["_rsi14"].shift(1)
df["prev_vol_ratio"] = df["volume"].shift(1) / df["_vol_ma5"].shift(1)

# Create missing sentiment aggregation features to resolve KeyError
df["sent_pos_mean"] = df["sent_pos"]
df["sent_neg_mean"] = df["sent_neg"]
df["sent_neu_mean"] = df["sent_neu"]
df["sent_pos_max"]  = df["sent_pos"]
df["sent_neg_max"]  = df["sent_neg"]
df["sent_variance"] = 0.0
df["n_headlines"]   = df["headline_count"]

FEATURE_COLS = [
    "sent_pos_mean", "sent_neg_mean", "sent_neu_mean",  # FinBERT mean    (3)
    "sent_pos_max",  "sent_neg_max",                    # FinBERT max     (2)
    "sent_variance", "n_headlines",                     # diversity       (2)
    "gap_open",                                         # price signal    (1)
    "prev_ma5_ratio", "prev_rsi14", "prev_vol_ratio",   # yesterday tech  (3)
]

df = df.dropna(subset=FEATURE_COLS + ["target"]).reset_index(drop=True)
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Rows after dropna: {len(df)}")

Features (11): ['sent_pos_mean', 'sent_neg_mean', 'sent_neu_mean', 'sent_pos_max', 'sent_neg_max', 'sent_variance', 'n_headlines', 'gap_open', 'prev_ma5_ratio', 'prev_rsi14', 'prev_vol_ratio']
Rows after dropna: 2557


In [7]:
# 4. Train / test split (chronological) + scale
split_index = int(len(df) * 0.8)

X_train = df[FEATURE_COLS].iloc[:split_index].values
y_train = df["target"].iloc[:split_index].values
X_test  = df[FEATURE_COLS].iloc[split_index:].values
y_test  = df["target"].iloc[split_index:].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Test target dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

Train: (2045, 11) | Test: (512, 11)
Test target dist: {np.int64(0): np.int64(238), np.int64(1): np.int64(274)}


In [8]:
# 5. Tabular Transformer — architecture
# Each scalar feature → independent 32-dim linear embedding (FT-Transformer style)
# CLS token prepended; 2-layer Transformer encoder; CLS output → binary classifier

D_MODEL      = 32
N_HEADS      = 2
N_LAYERS     = 2
FFN_DIM      = 64
DROPOUT      = 0.3
LR           = 3e-4
WEIGHT_DECAY = 1e-3
BATCH_SIZE   = 64
EPOCHS       = 150
PATIENCE     = 25

class FeatureTokenizer(nn.Module):
    """Each scalar feature x_i → x_i * w_i + b_i  (per-feature linear, shape → d_model)"""
    def __init__(self, n_features, d_model):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_model))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_model))
        nn.init.kaiming_uniform_(self.weight, a=0.01)

    def forward(self, x):
        # x: (B, n_features)  →  (B, n_features, d_model)
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class TabTransformer(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_layers, ffn_dim, dropout):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True,  # pre-norm: more stable on small data
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        tokens = self.tokenizer(x)                              # (B, F, d_model)
        cls    = self.cls_token.expand(x.size(0), -1, -1)      # (B, 1, d_model)
        tokens = torch.cat([cls, tokens], dim=1)                # (B, F+1, d_model)
        out    = self.encoder(tokens)
        return self.head(self.norm(out[:, 0])).squeeze(1)       # CLS output → logit

model = TabTransformer(
    n_features=len(FEATURE_COLS),
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    ffn_dim=FFN_DIM, dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")

TabTransformer(
  (tokenizer): FeatureTokenizer()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=64, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=64, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (head): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=

/tmp/ipykernel_2292/3551724967.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


In [9]:
# 6. Training loop with early stopping
X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
X_te = torch.tensor(X_test,  dtype=torch.float32)
y_te = torch.tensor(y_test,  dtype=torch.float32)

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)

criterion = nn.BCEWithLogitsLoss()   # classes ~balanced (53/47), no pos_weight needed
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float("inf")
patience_ctr  = 0
best_state    = None

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(X_tr)
    scheduler.step()

    # --- validate ---
    model.eval()
    with torch.no_grad():
        val_logits = model(X_te.to(device))
        val_loss   = criterion(val_logits, y_te.to(device)).item()
        val_preds  = (torch.sigmoid(val_logits) > 0.5).cpu().int().numpy()
        val_acc    = accuracy_score(y_test, val_preds)

    if epoch % 25 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr  = 0
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (patience={PATIENCE})")
            break

print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch   1 | train 0.6814 | val 0.6718 | acc 0.5293
Epoch  25 | train 0.6540 | val 0.6703 | acc 0.6055
Early stopping at epoch 30 (patience=25)

Best val loss: 0.6488


In [10]:
# 7. Evaluate best model
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    logits = model(X_te.to(device))
    probs  = torch.sigmoid(logits).cpu().numpy()
    preds  = (probs > 0.5).astype(int)

print("=== MultiSent Transformer (11 features, filtered) ===")
print(classification_report(y_test, preds))
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, probs), 4))

majority = y_test.sum() / len(y_test)
print("\n=====================================================")
print(f"  Random baseline (always up)               : {majority:.1%}")
print(f"  StockPredict   FinBERT→LinearSVC          : 64.1%")
print(f"  Combination    LinearSVC  (7 features)    : 64.2%")
print(f"  Combination    Transformer (7 features)   : 62.2%")
print(f"  MultiSent      Transformer (11 feat+filt) : {accuracy_score(y_test, preds):.1%}")
print("=====================================================")

=== MultiSent Transformer (11 features, filtered) ===
              precision    recall  f1-score   support

           0       0.61      0.67      0.64       238
           1       0.69      0.64      0.66       274

    accuracy                           0.65       512
   macro avg       0.65      0.65      0.65       512
weighted avg       0.65      0.65      0.65       512

Accuracy: 0.6504
ROC-AUC : 0.679

  Random baseline (always up)               : 53.5%
  StockPredict   FinBERT→LinearSVC          : 64.1%
  Combination    LinearSVC  (7 features)    : 64.2%
  Combination    Transformer (7 features)   : 62.2%
  MultiSent      Transformer (11 feat+filt) : 65.0%


In [11]:
# 8. Feature importance via input-gradient saliency
# grad * input magnitude tells us which features most influenced the predictions
model.eval()
X_sal = X_te.to(device).requires_grad_(True)
model(X_sal).sum().backward()

saliency = (X_sal.grad.abs() * X_sal.detach().abs()).mean(0).cpu().numpy()
fi = pd.Series(saliency, index=FEATURE_COLS).sort_values(ascending=False)
print("--- Feature saliency (gradient × input) ---")
print(fi.to_string())

--- Feature saliency (gradient × input) ---
gap_open          0.044998
sent_neu_mean     0.021035
prev_ma5_ratio    0.019132
sent_neg_mean     0.016767
sent_pos_max      0.009092
prev_rsi14        0.008576
sent_pos_mean     0.007361
sent_neg_max      0.007350
prev_vol_ratio    0.007333
n_headlines       0.006455
sent_variance     0.000000
